In [ ]:
# Notebook 3: Uplift modeling

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklift.metrics import perfect_qini_curve, qini_auc_score, qini_curve, uplift_at_k
from sklift.models import SoloModel, TwoModels

from hillstrom_uplift.data import load

df = load()


def plot_qini(y_true, uplift, treatment, label="model"):
    x, y = qini_curve(y_true, uplift, treatment)
    xp, yp = perfect_qini_curve(y_true, treatment)
    plt.plot(x, y, label=label)
    plt.plot(xp, yp, "--", color="gray", label="perfect")
    plt.plot([0, x[-1]], [0, y[-1]], ":", color="black", label="random")
    plt.xlabel("Số khách hàng được target")
    plt.ylabel("Incremental visits (Qini)")
    plt.legend()
    plt.show()


d = df[df.treatment.isin(["mens", "control"])].copy()
X = pd.get_dummies(
    d[["recency", "history", "mens", "womens", "zip_code", "newbie", "channel"]], drop_first=True
)
y = d["visit"]
trt = (d["treatment"] == "mens").astype(int)

X_tr, X_te, y_tr, y_te, t_tr, t_te = train_test_split(
    X, y, trt, test_size=0.3, random_state=42, stratify=trt
)

# S-learner: 1 mô hình, treatment là một feature
s = SoloModel(RandomForestClassifier(n_estimators=200, random_state=42))
s.fit(X_tr, y_tr, t_tr)

# T-learner: 2 mô hình riêng cho treated và control
t = TwoModels(
    estimator_trmnt=RandomForestClassifier(n_estimators=200, random_state=42),
    estimator_ctrl=RandomForestClassifier(n_estimators=200, random_state=42),
)
t.fit(X_tr, y_tr, t_tr)

for name, m in [("S-learner", s), ("T-learner", t)]:
    up = m.predict(X_te)
    print(
        name,
        "Qini AUC:",
        round(qini_auc_score(y_te, up, t_te), 4),
        "uplift@30%:",
        round(uplift_at_k(y_te, up, t_te, strategy="overall", k=0.3), 4),
    )

plot_qini(y_te, t.predict(X_te), t_te, label="T-learner")